# Week 12 — 落地与收尾：ONNX + FastAPI + 12 周总结

**目标**：把最佳 checkpoint 导出 ONNX，跑通 PyTorch / torch.compile / ONNX Runtime 三方延迟对比，做 INT8 量化 demo，最后起一个 FastAPI 推理服务并对外暴露。

**产出**：
- 三方延迟对比表（p50 / p95 / p99 over 500 runs，CPU + GPU）
- ONNX 数值一致性验证（max abs diff < 1e-4）
- INT8 动态量化 before/after（延迟 + AUC-PR）
- `server.py` + `pyngrok` 对外 URL + `curl` 示例
- Production readiness 清单 + 12 周 Mermaid 全景图

In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')

## 1. 依赖安装 + 加载最佳 checkpoint（W10 或 W11，带 graceful fallback）

W11 的 PatchTST 训练稳定、导 ONNX 算子更干净，所以优先用 `w11_patchtst.pt`。找不到时退回 W10（`w10_dualhead.pt`）或 W11 vanilla。再没有的话，实例化一个 PatchTST 兜底，保证本 notebook 端到端可跑。

In [ ]:
!pip install -q onnx==1.15.0 onnxruntime==1.17.0 fastapi==0.110.0 \
    uvicorn==0.27.1 pyngrok==7.1.5 pydantic==2.6.1

In [ ]:
import math, time, json
import numpy as np, torch
import torch.nn as nn

# ---- minimal inline PatchTST (same as W11) ----
class PatchEmbed(nn.Module):
    def __init__(self, patch_len, stride, d_model):
        super().__init__()
        self.patch_len, self.stride = patch_len, stride
        self.proj = nn.Linear(patch_len, d_model)
    def forward(self, x):
        x = x.squeeze(-1)
        x = x.unfold(-1, self.patch_len, self.stride)
        return self.proj(x)

class PatchTST(nn.Module):
    def __init__(self, n_feats, seq_len, patch_len=16, stride=8,
                 d_model=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.n_feats, self.patch_len, self.stride = n_feats, patch_len, stride
        n_patches = (seq_len - patch_len) // stride + 1
        self.embed = PatchEmbed(patch_len, stride, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, n_patches, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Linear(d_model, 1))

    def forward(self, x):
        B, L, F_ = x.shape
        x = x.permute(0, 2, 1).reshape(B * F_, L, 1)
        x = self.embed(x) + self.pos_emb
        x = self.encoder(x)
        x = x.mean(dim=1)
        logit = self.head(x).squeeze(-1).view(B, F_)
        return logit.mean(dim=1)


def load_best_checkpoint():
    ckpt_dir = PROJECT_ROOT / 'checkpoints'
    candidates = ['w11_patchtst.pt', 'w10_dualhead.pt', 'w11_vanilla.pt']
    for name in candidates:
        p = ckpt_dir / name
        if p.exists():
            print(f'[load] found {p}')
            data = torch.load(p, map_location='cpu')
            cfg = data.get('config', dict(n_feats=8, seq_len=256))
            cfg.setdefault('patch_len', 16); cfg.setdefault('stride', 8)
            cfg.setdefault('d_model', 64); cfg.setdefault('n_heads', 4)
            cfg.setdefault('n_layers', 3)
            m = PatchTST(**{k: cfg[k] for k in
                ('n_feats','seq_len','patch_len','stride','d_model','n_heads','n_layers')})
            try:
                m.load_state_dict(data['state_dict'], strict=False)
                print('[load] state_dict loaded (strict=False)')
            except Exception as e:
                print(f'[load] state_dict mismatch, using random init: {e}')
            return m.eval(), cfg
    print('[load] no checkpoint found, using random-init PatchTST')
    cfg = dict(n_feats=8, seq_len=256, patch_len=16, stride=8,
               d_model=64, n_heads=4, n_layers=3)
    return PatchTST(**cfg).eval(), cfg


model_cpu, CFG = load_best_checkpoint()
N_FEATS, SEQ_LEN = CFG['n_feats'], CFG['seq_len']
print('config:', CFG)
print('params:', sum(p.numel() for p in model_cpu.parameters()))

## 2. torch.compile 基准：100 次 forward，对比 eager vs compiled

`torch.compile` 默认模式是 `default`（graph capture + 融合算子 + Triton kernel）。第一次 forward 会慢（JIT），所以 warmup 单独跑。

In [ ]:
def bench_forward(fn, x, n_iter=100, warmup=10, use_cuda=False):
    for _ in range(warmup):
        fn(x)
    if use_cuda: torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        fn(x)
    if use_cuda: torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_iter


@torch.no_grad()
def eager_call(x): return model_gpu(x)

if torch.cuda.is_available():
    model_gpu = model_cpu.to('cuda').eval()
    x_gpu = torch.randn(1, SEQ_LEN, N_FEATS, device='cuda')
    eager_ms = bench_forward(eager_call, x_gpu, use_cuda=True) * 1000
    try:
        compiled = torch.compile(model_gpu, mode='default')
        @torch.no_grad()
        def compiled_call(x): return compiled(x)
        comp_ms = bench_forward(compiled_call, x_gpu, use_cuda=True) * 1000
    except Exception as e:
        print(f'[compile] failed on this env: {e}')
        comp_ms = float('nan')
    print(f'eager      p50 ~ {eager_ms:.3f} ms / forward')
    print(f'compiled   p50 ~ {comp_ms:.3f} ms / forward')
    if comp_ms == comp_ms:
        print(f'speedup    x{eager_ms/comp_ms:.2f}')
else:
    model_gpu = None
    eager_ms = comp_ms = float('nan')
    print('[compile] skipped — no CUDA available')

## 3. ONNX 导出 + 数值一致性校验

- `dynamic_axes` 放在 batch dim，让 ORT 支持变长 batch。
- 用 32 个随机 batch 对比 PyTorch vs ORT 的输出，要求 `max|Δ| < 1e-4`。

In [ ]:
import onnx, onnxruntime as ort

model_cpu = model_cpu.cpu().eval()
onnx_path = PROJECT_ROOT / 'models' / 'transformer.onnx'
onnx_path.parent.mkdir(parents=True, exist_ok=True)

dummy = torch.randn(1, SEQ_LEN, N_FEATS)
torch.onnx.export(
    model_cpu, dummy, str(onnx_path),
    input_names=['x'], output_names=['score'],
    dynamic_axes={'x': {0: 'batch'}, 'score': {0: 'batch'}},
    opset_version=17,
    do_constant_folding=True,
)
onnx.checker.check_model(onnx.load(str(onnx_path)))
print('ONNX saved at', onnx_path, '(', onnx_path.stat().st_size / 1024, 'KB )')

In [ ]:
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])

max_diffs = []
with torch.no_grad():
    for _ in range(32):
        bs = int(np.random.randint(1, 16))
        x_np = np.random.randn(bs, SEQ_LEN, N_FEATS).astype(np.float32)
        y_pt = model_cpu(torch.from_numpy(x_np)).numpy()
        y_ort = sess.run(None, {'x': x_np})[0]
        max_diffs.append(np.abs(y_pt - y_ort).max())

print(f'max abs diff over 32 batches: {max(max_diffs):.2e}')
assert max(max_diffs) < 1e-4, 'numerical mismatch beyond 1e-4 — investigate opset / dtype'
print('OK: PyTorch and ORT agree within 1e-4')

## 4. 延迟基准：PyTorch eager / torch.compile / ONNX Runtime（CPU + GPU）

每种后端跑 500 次 forward（batch=1），统计 p50 / p95 / p99。GPU 不可用时自动跳过 GPU 行。

In [ ]:
import pandas as pd

def percentiles(samples_ms):
    a = np.asarray(samples_ms)
    return {
        'p50': float(np.percentile(a, 50)),
        'p95': float(np.percentile(a, 95)),
        'p99': float(np.percentile(a, 99)),
    }

def time_many(fn, x, n=500, warmup=20, use_cuda=False):
    for _ in range(warmup):
        fn(x)
    if use_cuda: torch.cuda.synchronize()
    out = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn(x)
        if use_cuda: torch.cuda.synchronize()
        out.append((time.perf_counter() - t0) * 1000)
    return out

rows = []

# ---- CPU: PyTorch eager ----
x_cpu_np = np.random.randn(1, SEQ_LEN, N_FEATS).astype(np.float32)
x_cpu    = torch.from_numpy(x_cpu_np)
with torch.no_grad():
    samples = time_many(lambda x: model_cpu(x), x_cpu)
rows.append({'backend': 'PyTorch eager', 'device': 'CPU', **percentiles(samples)})

# ---- CPU: ONNX Runtime ----
sess_cpu = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
samples = time_many(lambda x: sess_cpu.run(None, {'x': x}), x_cpu_np)
rows.append({'backend': 'ONNX Runtime', 'device': 'CPU', **percentiles(samples)})

# ---- GPU: PyTorch eager ----
if torch.cuda.is_available():
    x_gpu = torch.randn(1, SEQ_LEN, N_FEATS, device='cuda')
    with torch.no_grad():
        samples = time_many(lambda x: model_gpu(x), x_gpu, use_cuda=True)
    rows.append({'backend': 'PyTorch eager', 'device': 'GPU', **percentiles(samples)})

    # torch.compile
    try:
        compiled = torch.compile(model_gpu, mode='default')
        with torch.no_grad():
            samples = time_many(lambda x: compiled(x), x_gpu, use_cuda=True)
        rows.append({'backend': 'torch.compile', 'device': 'GPU', **percentiles(samples)})
    except Exception as e:
        print(f'[compile] skipped: {e}')

    # ORT CUDAExecutionProvider if available
    available = ort.get_available_providers()
    if 'CUDAExecutionProvider' in available:
        sess_gpu = ort.InferenceSession(str(onnx_path),
                    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        x_np_gpu = np.random.randn(1, SEQ_LEN, N_FEATS).astype(np.float32)
        samples = time_many(lambda x: sess_gpu.run(None, {'x': x}), x_np_gpu)
        rows.append({'backend': 'ONNX Runtime', 'device': 'GPU', **percentiles(samples)})
    else:
        print('[ort] CUDAExecutionProvider not available, skipped')

lat_tbl = pd.DataFrame(rows)
lat_tbl

## 5. INT8 动态量化 demo

把 `nn.Linear` 动态量化成 INT8（激活仍是 FP32，只量化权重 + 动态 per-batch 激活 scale）。最省事的一档，通常能在 CPU 上拿到 2-3x 加速。这里给出：

- 延迟对比（CPU p50）
- AUC-PR 对比（在随机生成的验证集上——真实项目应用 W10/W11 的真正 val loader）

In [ ]:
from torch.quantization import quantize_dynamic
from sklearn.metrics import average_precision_score

# Build a small synthetic eval set (stand-in; in a real project use W10/W11 val loader)
N_EVAL = 400
fr_rate = 0.08
rng = np.random.default_rng(0)
ys = (rng.random(N_EVAL) < fr_rate).astype(np.int64)
X_eval = rng.normal(0, 1, (N_EVAL, SEQ_LEN, N_FEATS)).astype(np.float32)
# inject mild signal into 'fraud' examples so AUC-PR is non-trivial
for i, y in enumerate(ys):
    if y:
        t0 = rng.integers(0, SEQ_LEN - 16)
        X_eval[i, t0:t0+16, :] += rng.uniform(0.5, 1.5)

def eval_aucpr(m, X, y, batch=32):
    m.eval(); scores = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.from_numpy(X[i:i+batch])
            scores.append(torch.sigmoid(m(xb)).numpy())
    return average_precision_score(y, np.concatenate(scores))

aucpr_fp32 = eval_aucpr(model_cpu, X_eval, ys)

q_model = quantize_dynamic(model_cpu, {nn.Linear}, dtype=torch.qint8)
aucpr_int8 = eval_aucpr(q_model, X_eval, ys)

x_cpu = torch.from_numpy(X_eval[:1])
with torch.no_grad():
    lat_fp32 = np.median(time_many(lambda x: model_cpu(x), x_cpu, n=200))
    lat_int8 = np.median(time_many(lambda x: q_model(x),   x_cpu, n=200))

pd.DataFrame([
    {'variant': 'FP32', 'CPU_p50_ms': round(lat_fp32, 3), 'AUC_PR': round(aucpr_fp32, 4)},
    {'variant': 'INT8 dynamic', 'CPU_p50_ms': round(lat_int8, 3), 'AUC_PR': round(aucpr_int8, 4)},
])

## 6. FastAPI 推理服务 + pyngrok 对外暴露

三步走：
1. **写 `server.py`**：加载 ONNX 模型，定义 Pydantic schema，暴露 `POST /score`。
2. **起 tunnel**：`pyngrok` 把本地 `uvicorn` 映射到公网 URL。
3. **`curl` 验证**：demo 请求 + 期望返回。

> 真正启动 `uvicorn` 是阻塞的，所以留一段 **已注释**的命令；Colab 用户复制出去手动跑。这样本 notebook 不会挂死在最后一格。

In [ ]:
server_code = '''"""Minimal FastAPI inference server for the Transformer anomaly model.

Start with:  uvicorn server:app --host 0.0.0.0 --port 8000
"""
import os, numpy as np, onnxruntime as ort
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List

ONNX_PATH = os.environ.get("ONNX_PATH", "models/transformer.onnx")
SEQ_LEN   = int(os.environ.get("SEQ_LEN", "256"))
N_FEATS   = int(os.environ.get("N_FEATS", "8"))

sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

app  = FastAPI(title="Transformer Anomaly Service", version="0.5")


class ScoreRequest(BaseModel):
    # shape: (batch, seq_len, n_feats)
    sequences: List[List[List[float]]] = Field(..., description="(B, L, F) float tensor")


class ScoreResponse(BaseModel):
    scores: List[float]
    model:  str = "patchtst-v0.5"


@app.get("/health")
def health():
    return {"status": "ok", "onnx": ONNX_PATH}


@app.post("/score", response_model=ScoreResponse)
def score(req: ScoreRequest):
    x = np.asarray(req.sequences, dtype=np.float32)
    if x.ndim != 3 or x.shape[1] != SEQ_LEN or x.shape[2] != N_FEATS:
        raise HTTPException(400, f"expect shape (B, {SEQ_LEN}, {N_FEATS}), got {x.shape}")
    logits = sess.run(None, {"x": x})[0]
    probs  = 1.0 / (1.0 + np.exp(-logits))
    return ScoreResponse(scores=probs.tolist())
'''

(PROJECT_ROOT / 'server.py').write_text(server_code)
print('wrote', PROJECT_ROOT / 'server.py')

In [ ]:
# Show the code so the learner sees what they just wrote
print((PROJECT_ROOT / 'server.py').read_text())

In [ ]:
# --- pyngrok tunnel (Colab) ---
# Requires NGROK_AUTH_TOKEN env var (or Colab Secret). Uncomment to run.
#
# from pyngrok import ngrok, conf
# import subprocess, time
# os.environ.setdefault('ONNX_PATH', str(PROJECT_ROOT / 'models' / 'transformer.onnx'))
# os.environ.setdefault('SEQ_LEN',   str(SEQ_LEN))
# os.environ.setdefault('N_FEATS',   str(N_FEATS))
# token = os.environ.get('NGROK_AUTH_TOKEN') or userdata.get('NGROK_AUTH_TOKEN')
# conf.get_default().auth_token = token
# proc = subprocess.Popen(['uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
#                         cwd=str(PROJECT_ROOT))
# time.sleep(3)
# tunnel = ngrok.connect(8000)
# print('Public URL:', tunnel.public_url)
#
# # curl example (run in another cell / terminal):
# # curl -X POST $PUBLIC_URL/score -H 'Content-Type: application/json' \
# #      -d '{"sequences": [[[0.0]*8]*256]}'
#
# # Shutdown when done:
# # ngrok.disconnect(tunnel.public_url); proc.terminate()

print('server.py written; uncomment block above in Colab with NGROK_AUTH_TOKEN set.')

In [ ]:
# Sanity-check server.py locally by calling the scoring code path directly
# (without actually starting uvicorn — avoids blocking this notebook).
import importlib.util, sys
spec = importlib.util.spec_from_file_location('server', PROJECT_ROOT / 'server.py')
server = importlib.util.module_from_spec(spec)
os.environ['ONNX_PATH'] = str(PROJECT_ROOT / 'models' / 'transformer.onnx')
os.environ['SEQ_LEN']   = str(SEQ_LEN)
os.environ['N_FEATS']   = str(N_FEATS)
spec.loader.exec_module(server)

req = server.ScoreRequest(sequences=np.random.randn(2, SEQ_LEN, N_FEATS).tolist())
resp = server.score(req)
print('local call:', resp)

## 7. Production readiness 清单

每一项都应在上线前闭环：

| # | 维度 | 关键问题 | 本项目的当前状态 |
|---|------|----------|------------------|
| 1 | **特征一致性** | 训练用的特征处理逻辑能 1:1 复用到线上吗？scaler 的 `mu/std` 是否持久化？ | 训练时 `mu, std` 仅存在内存，需序列化到 `models/scaler.json` |
| 2 | **Schema 校验** | 请求体是否校验 shape / dtype / 范围？ | 已有 Pydantic + FastAPI 基础校验，但范围/缺失值校验待补 |
| 3 | **模型漂移监控** | 输入分布漂移、标签漂移、PSI/KS；预测分数分布是否稳定 | 需接入离线监控（每日跑 PSI）+ 线上分数直方图 |
| 4 | **A/B 发布** | 流量切分、guardrail（AUC-PR 下降阈值）、一键回滚 | 还没做；建议用 shadow traffic 起步 |
| 5 | **概率校准** | 分数是不是可比/可阈值化？Platt / Isotonic | 未校准；线上建议加 Platt scaling |
| 6 | **可解释性** | 单个 case 为什么被判为欺诈？attention map / IG / SHAP | attention map 容易可视化，Platt + 局部 IG 更稳 |
| 7 | **延迟/吞吐 SLA** | p99 < 多少 ms？QPS 上限？ | p50/p95/p99 已基准；SLA 待和业务侧对齐 |
| 8 | **安全/合规** | API auth、PII 脱敏、审计日志 | 本 demo 无 auth，生产必须加 |

## 8. 12 周全景图（Mermaid）

```mermaid
flowchart LR
    A["Week 1-2 数据理解 + 基线"] --> B["Week 3 MLP v0.1"]
    B --> C["Week 4 LSTM v0.2"]
    C --> D["Week 5-6 Transformer v0.3 分类"]
    D --> E["Week 7-8 Transformer v0.3.1 Kaggle 真实数据"]
    E --> F["Week 9 无监督重构 v0.4 上"]
    F --> G["Week 10 双头 分类 + 重构 v0.4 下"]
    G --> H["Week 11 PatchTST v0.5 长序列"]
    H --> I["Week 12 ONNX + FastAPI 上线"]

    subgraph "产出"
      I --> J1["ONNX 模型"]
      I --> J2["FastAPI 服务"]
      I --> J3["12 周博客"]
    end
```

## 9. 最后一次复盘

- 12 周走下来，你从「MLP 看数据」一路走到「PatchTST + ONNX 服务化」，每一周都有一个小 MVP 落地。
- 三张表是你这一期最有价值的资产：各 baseline 对比表（W7-W10）、长序列三方对比表（W11）、三后端延迟表（W12）——把这些数字直接复制进 `12_summary_blog.md`。
- 下一步方向（在博客里展开）：
  1. 对比学习预训练（SimCLR / TS2Vec 风格）再微调；
  2. 图 Transformer：把用户-商户-设备建成异构图，用 Graph Transformer 统一建模；
  3. LLM-augmented：用 LLM 生成异常推理链路，辅助人工审核 / 加速规则迭代。